In [1]:
from scipy import stats
import numpy as np

In [17]:
import pandas as pd

def spearman_distance(list1, list2, verbose=False):
    """
    Calculates the Spearman distance between two ranked lists with potentially
    mismatched items.

    Args:
        list1 (list): The first ranked list of preferences.
        list2 (list): The second ranked list of preferences.
        verbose (bool): If True, prints the intermediate rank table for clarity.

    Returns:
        float: The calculated distance, where 0.0 is identical.
    """
    # 1. Find the union of all preferences
    all_items = set(list1) | set(list2)
    n = len(all_items)

    # If there's no overlap or lists are too small, correlation is undefined.
    # We can define distance as maximal in this case.
    if n <= 1:
        return 2.0  # Max distance

    # 2. Create rank dictionaries for each list
    # The rank is the index + 1
    ranks1 = {item: i + 1 for i, item in enumerate(list1)}
    ranks2 = {item: i + 1 for i, item in enumerate(list2)}

    # 3. Calculate sum of squared differences (d^2)
    sum_sq_diff = 0
    rank_data = []

    # Use the union of items as the basis for comparison
    for item in all_items:
        # Assign a penalty rank (n + 1) if an item is missing
        rank1 = ranks1.get(item, n + 1)
        rank2 = ranks2.get(item, n + 1)
        diff = rank1 - rank2
        sum_sq_diff += diff ** 2
        rank_data.append([item, rank1, rank2, diff])

    if verbose:
        rank_df = pd.DataFrame(rank_data, columns=["Item", "Rank 1", "Rank 2", "Difference"])
        print(rank_df.to_string(index=False), "\n")

    # 4. Calculate Spearman's rank correlation coefficient (rho)
    # ρ = 1 - (6 * Σd²) / (n * (n² - 1))
    rho = 1 - (6 * sum_sq_diff) / (n * (n**2 - 1))

    # 5. Convert correlation to distance
    # Distance = 1 - ρ
    distance = 1 - rho

    return distance

# --- DEMONSTRATION ---

# Our existing dataset of entities and their preferences
entities = {
    "Entity 1": ["B", "Q", "A"],
    "Entity 2": ["A", "B", "C"],
    "Entity 3": ["B", "A", "Q"], # Very similar to Entity 1
    "Entity 4": ["X", "Y", "Z"], # Very different from the new entity
    "Entity 5": ["A", "C", "B"]  # Same items as new entity, different order
}

# The new entity we want to find neighbors for
new_entity_name = "New Entity"
new_entity_prefs = ["B", "A", "C"]

print(f"Finding closest neighbors for '{new_entity_name}': {new_entity_prefs}\n")
print("--- Detailed Calculation for Entity 3 vs New Entity ---")
spearman_distance(entities["Entity 3"], new_entity_prefs, verbose=True)


# --- Brute-Force Search ---
# Calculate the distance from the new entity to all existing entities
distances = []
for name, prefs in entities.items():
    dist = spearman_distance(prefs, new_entity_prefs)
    distances.append((name, dist))

# Sort the entities by distance to find the closest ones
distances.sort(key=lambda x: x[1])

print("--- Top N Closest Neighbors ---")
for name, dist in distances:
    print(f"{name:<12} | Distance: {dist:.4f}")

Finding closest neighbors for 'New Entity': ['B', 'A', 'C']

--- Detailed Calculation for Entity 3 vs New Entity ---
Item  Rank 1  Rank 2  Difference
   B       1       1           0
   A       2       2           0
   Q       3       5          -2
   C       5       3           2 

--- Top N Closest Neighbors ---
Entity 2     | Distance: 0.5000
Entity 3     | Distance: 0.8000
Entity 1     | Distance: 1.4000
Entity 5     | Distance: 1.5000
Entity 4     | Distance: 4.4000


In [16]:
def kendall_tau_to_distance(tau_coefficient):
    return (1 - tau_coefficient) / 2

# Using the tau_coefficient from the previous step
distance = kendall_tau_to_distance(tau_coefficient)
print(distance)

1.0
